In [ ]:


import numpy as np
import pandas as pd



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/original_Macular Scar1081_79.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/original_Macular Scar1063_732.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/augmented_Macular Scar1780_1400.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/original_Macular Scar1922_651.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/augmented_Macular Scar1570_2378.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/augmented_Macular Scar1379_1839.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6/val/Macular Scar/augmented_Macular Scar1579_1716.jpg
/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggl

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [ ]:
# Hyperparameters
IMAGE_SIZE = 224
NUM_CLASSES = 6
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.0005
PATIENCE = 5


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
dataset_dir = "/kaggle/input/split-dataset-augmented-kaggle-6/split_dataset_augmented_kaggle_6"
train_dir = f"{dataset_dir}/train"
val_dir = f"{dataset_dir}/val"
test_dir = f"{dataset_dir}/test"

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
base_model = timm.create_model("vgg16_bn", pretrained=True, num_classes=0)
base_model = base_model.to(DEVICE)


model.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

In [ ]:
class CustomVGG(nn.Module):
    def __init__(self, base_model, num_classes=6):
        super(CustomVGG, self).__init__()
        self.base_model = base_model
        self.additional_layers = nn.Sequential(
            nn.Linear(4096, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.base_model(x)
        x = self.additional_layers(x)
        return x

# Initialize Model
model = CustomVGG(base_model, num_classes=NUM_CLASSES).to(DEVICE)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
early_stopping_counter = 0
best_val_loss = float('inf')


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = total_loss / len(train_loader)
    train_acc = 100 * correct / total

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * correct_val / total_val

    print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_vgg16_bn_custom.pth")
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= PATIENCE:
        print("Early stopping triggered!")
        break

    scheduler.step()


Epoch [1/15], Train Loss: 0.7525, Train Acc: 67.85%, Val Loss: 0.5196, Val Acc: 78.53%
Epoch [2/15], Train Loss: 0.4920, Train Acc: 80.58%, Val Loss: 0.5605, Val Acc: 78.07%
Epoch [3/15], Train Loss: 0.4032, Train Acc: 84.45%, Val Loss: 0.3285, Val Acc: 85.73%
Epoch [4/15], Train Loss: 0.3884, Train Acc: 85.08%, Val Loss: 0.4323, Val Acc: 79.33%
Epoch [5/15], Train Loss: 0.3433, Train Acc: 87.54%, Val Loss: 0.3495, Val Acc: 86.67%
Epoch [6/15], Train Loss: 0.2738, Train Acc: 89.95%, Val Loss: 0.2791, Val Acc: 88.00%
Epoch [7/15], Train Loss: 0.2194, Train Acc: 92.19%, Val Loss: 0.2688, Val Acc: 89.60%
Epoch [8/15], Train Loss: 0.1970, Train Acc: 93.19%, Val Loss: 0.2294, Val Acc: 91.53%
Epoch [9/15], Train Loss: 0.1661, Train Acc: 94.02%, Val Loss: 0.1840, Val Acc: 93.20%
Epoch [10/15], Train Loss: 0.1235, Train Acc: 95.66%, Val Loss: 0.1849, Val Acc: 93.07%
Epoch [11/15], Train Loss: 0.0954, Train Acc: 96.70%, Val Loss: 0.1676, Val Acc: 93.53%
Epoch [12/15], Train Loss: 0.0853, Train 

In [ ]:
import torch
from sklearn.metrics import precision_score, recall_score, f1_score

model.load_state_dict(torch.load("best_vgg16_bn_custom.pth"))
model.eval()

correct_test = 0
total_test = 0
all_test_labels = []
all_test_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_preds.extend(predicted.cpu().numpy())

# Compute accuracy
test_acc = 100 * correct_test / total_test

# Compute precision, recall, and F1-score
test_precision = precision_score(all_test_labels, all_test_preds, average='weighted', zero_division=1)
test_recall = recall_score(all_test_labels, all_test_preds, average='weighted', zero_division=1)
test_f1 = f1_score(all_test_labels, all_test_preds, average='weighted', zero_division=1)

# Get total parameters in the model
total_params = sum(p.numel() for p in model.parameters())

# Print results
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test Precision: {test_precision:.4f}, Test Recall: {test_recall:.4f}, Test F1-Score: {test_f1:.4f}")
print(f"Total Parameters: {total_params / 1e6:.2f}M")
print("Training and evaluation completed!")

<ipython-input-10-3d1fc208fe9a>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_vgg16_bn_custom.pth"))


Test Accuracy: 96.47%
Test Precision: 0.9647, Test Recall: 0.9647, Test F1-Score: 0.9647
Total Parameters: 136.50M
Training and evaluation completed!
